# Course 1: From Beginner to an AI Research Lab
## Session 11: Writing and sharing research

**Goal:** write the contributions first, make a figure that shows every run, and keep a four-page draft to its page budget.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a GPU, a download or an account. Cells that are worksheets hold example rows for you to replace with your own.

### 1. Contributions first
Two or three bullets, each naming the figure or table that will support it. A bullet with no evidence is your next experiment, or a bullet to delete.

In [ ]:
import pandas as pd
contributions = pd.DataFrame([
    {"contribution": "<We show that ... on ...>", "supported_by": "<Table 1>", "status": "have the result"},
    {"contribution": "<We find that removing X costs ...>", "supported_by": "<Figure 2>", "status": "have the result"},
    {"contribution": "<We release code and ...>", "supported_by": "", "status": "TODO"},
])
missing = contributions[contributions.supported_by == ""]
print(contributions.to_string(index=False)); print("contributions with no evidence yet:", len(missing))

### 2. Figures before prose
A figure checklist:
- one message, stated in the caption,
- axes labelled, with units,
- error bars or all runs shown,
- readable in grey and at column width,
- the number of seeds given in the caption.

Compare a figure straight from a notebook with one made for a paper.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy import stats
rng = np.random.default_rng(0)
methods = {"Baseline": 70.0 + rng.normal(0, 0.6, 5), "Ours": 71.0 + rng.normal(0, 0.6, 5), "Ours, no X": 70.5 + rng.normal(0, 0.6, 5)}

fig, ax = plt.subplots(); ax.bar(list(methods), [v.mean() for v in methods.values()]); fig.savefig("figure_notebook.png")     # the notebook version

plt.rcParams.update({"font.size": 8})
fig, ax = plt.subplots(figsize=(3.3, 2.2))                                   # one column of a two-column paper is about 3.3 inches wide
markers = ["o", "s", "^"]
for i, (name, v) in enumerate(methods.items()):
    half = stats.t.ppf(0.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v))
    ax.plot([i] * len(v), v, markers[i], color="0.55", markersize=4, zorder=2)                 # every run
    ax.errorbar([i + 0.18], [v.mean()], yerr=[half], fmt=markers[i], color="black", capsize=3, markersize=5, zorder=3)   # mean and 95% interval
ax.set_xticks(range(len(methods))); ax.set_xticklabels(list(methods)); ax.set_ylabel("Test accuracy (%)")
ax.spines[["top", "right"]].set_visible(False); fig.tight_layout(); fig.savefig("figure_paper.pdf"); fig.savefig("figure_paper.png", dpi=300)
CAPTION = "Figure 1: Our method improves test accuracy over the baseline; removing X gives back about half of the gain. Grey points are individual runs (5 seeds each); black markers show the mean and 95% t-interval."
print(CAPTION)

What changed: the y-axis no longer starts at zero to exaggerate nothing and hide everything, every run is visible, the interval is shown, methods differ by marker shape so the figure works in grey, the fonts are readable at column width, and the caption states the message and the number of seeds. Save vector files (PDF) for the paper.

### 3. The page budget for four pages
```
Abstract        150 words
Introduction    0.75 page
Method, setup   0.75 page
Experiments     1.5 pages
Related work    0.5 page
Limitations     0.25 page
Conclusion      0.25 page
References and appendix do not count
```
A rough conversion: one page of a two-column conference template holds about 800 words of plain text, fewer with figures. Check your template.

In [ ]:
WORDS_PER_PAGE = 800                                # an estimate; measure it on your own template
BUDGET = {"Abstract": 150, "Introduction": 0.75 * WORDS_PER_PAGE, "Method and setup": 0.75 * WORDS_PER_PAGE, "Experiments": 1.5 * WORDS_PER_PAGE,
          "Related work": 0.5 * WORDS_PER_PAGE, "Limitations": 0.25 * WORDS_PER_PAGE, "Conclusion": 0.25 * WORDS_PER_PAGE}

def check_draft(text):
    """Draft in markdown with '## Section' headings. Returns words per section against the budget."""
    sections, current = {}, None
    for line in text.splitlines():
        if line.startswith("## "): current = line[3:].strip(); sections[current] = 0
        elif current: sections[current] += len(line.split())
    rows = [{"section": s, "words": sections.get(s, 0), "budget": int(b), "over_by": max(0, sections.get(s, 0) - int(b))} for s, b in BUDGET.items()]
    return pd.DataFrame(rows)

draft = "## Abstract\n" + "word " * 190 + "\n## Introduction\n" + "word " * 500 + "\n## Method and setup\n" + "word " * 700 + "\n## Experiments\n" + "word " * 900
report = check_draft(draft); print(report.to_string(index=False))
assert report.loc[report.section == "Abstract", "over_by"].item() == 40

The first draft runs long. Cutting to four pages is where a paper becomes clear. Write the abstract last.

### 4. An abstract in five sentences
1. The problem and why it matters.
2. What is missing in current work.
3. What you did.
4. The main result, with a number.
5. What it means, and one limit.

In [ ]:
def abstract_check(text):
    words = len(text.split()); has_number = any(ch.isdigit() for ch in text)
    vague = [w for w in ["novel", "state-of-the-art", "significantly", "very", "extensive"] if w in text.lower()]
    return {"words": words, "within_150": words <= 150, "states_a_number": has_number, "words_to_reconsider": vague}
print(abstract_check("We propose a novel method that significantly improves results on extensive benchmarks."))
print(abstract_check("On a 10M-parameter Bangla language model, replacing learned positions with rotary positions lowers validation loss from 1.52 to 1.47 (5 seeds, 95% interval ±0.02) at equal training steps."))

'Significantly' has a statistical meaning. Use it only with a test behind it, or drop it and give the number.

### 5. Review a classmate's paper in the conference format
A summary, three strengths, three weaknesses, questions, and a score with your confidence. Then write a response to your own reviews: each point raised, and what you changed.

In [ ]:
REVIEW = """Summary (what the paper claims and shows, in your own words):
Strengths:
  1.
  2.
  3.
Weaknesses (each one specific and checkable):
  1.
  2.
  3.
Questions for the authors:
Score (1-10) and confidence (1-5):
"""
RESPONSE = pd.DataFrame([{"reviewer_point": "<W1: only 3 seeds>", "what_i_changed": "<ran 5 seeds; Table 1 updated>", "where": "<Section 4>"}])
print(REVIEW); print(RESPONSE.to_string(index=False))

### 6. Sharing, and AI writing tools
- Where: a workshop, a student track, arXiv (check the current endorsement rules for your category), or a well-documented repository with a short report.
- AI tools: venues differ and their rules change every year. Read the current policy of the venue you submit to. The constant across venues is that the authors are responsible for every sentence and every citation. Do not include text you cannot defend, or a reference you have not read.

### 7. A five-minute talk
One idea and one figure per slide, five slides at most.

### TODO
1. Write your contributions list and make your main figure with the checklist.
2. Draft to the page budget and run `check_draft` on it.
3. Review a classmate's draft, revise yours, and hand in the paper with the response table.